In [ ]:
"""S3 utils"""
import csv
from datetime import datetime
from typing import Any
import boto3
import polars as pl
import smart_open
from models.events import IncrementalWindow

S3 = boto3.Session(profile_name="platform-developer").client("s3")
TRANSPORT_PARAMS = {"client": S3}


def df_from_s3_parquet(s3_uri: str) -> pl.DataFrame:
    with smart_open.open(s3_uri, "rb", transport_params=TRANSPORT_PARAMS) as f:
        return pl.read_parquet(f)


def get_csv_from_s3(s3_uri: str) -> list[Any]:
    with smart_open.open(s3_uri, "r", transport_params=TRANSPORT_PARAMS) as f:
        return list(csv.DictReader(f))


def list_s3_uris(bucket_name: str, prefix: str, suffix: str) -> list[str]:
    """
    Return the S3 URIs of all objects which match the selected prefix and suffix.
    """
    paginator = S3.get_paginator("list_objects_v2")
    page_iterator = paginator.paginate(Bucket=bucket_name, Prefix=prefix)

    files = []
    for page in page_iterator:
        for item in page.get("Contents", []):
            if item["Key"].endswith(suffix):
                files.append(f"s3://{bucket_name}/{item['Key']}")

    return files


def list_incremental_windows(
        bucket_name: str,
        prefix: str,
        range_start: datetime | None = None,
        range_end: datetime | None = None,
) -> list[IncrementalWindow]:
    """
    Return a list of incremental time windows which exist at the specified S3 prefix.
    Each time window is a path segment in the format `YYYYMMDD'T'HHmm-YYYYMMDD'T'HHmm`.
    """
    windows = []
    paginator = S3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix, Delimiter="/"):
        for common_prefix in page["CommonPrefixes"]:
            full_prefix = common_prefix["Prefix"]
            raw_window = full_prefix[len(prefix):].rstrip("/")

            try:
                window = strp_window(raw_window)
            except ValueError:
                print(f"Could not parse prefix '{raw_window}' into a time window.")
                continue

            starts_after_range_start = not range_start or window.start_time >= range_start
            ends_before_range_end = not range_end or window.end_time <= range_end
            if starts_after_range_start and ends_before_range_end:
                windows.append(window)

    return windows

In [2]:
"""Incremental window utils"""

def strf_window(window: IncrementalWindow) -> str:
    return f"{window.start_time.strftime('%Y%m%dT%H%M')}-{window.end_time.strftime('%Y%m%dT%H%M')}"


def strp_window(window: str) -> IncrementalWindow:
    raw_start, raw_end = window.split("-")
    window_start = datetime.strptime(raw_start, "%Y%m%dT%H%M")
    window_end = datetime.strptime(raw_end, "%Y%m%dT%H%M")
    return IncrementalWindow(start_time=window_start, end_time=window_end)

In [3]:
"""S3 Reader classes"""

from concurrent.futures import ThreadPoolExecutor
from typing import Literal

from pydantic import BaseModel

from utils.types import CatalogueTransformerType, EntityType, TransformerType

S3_PARALLELISM = 10
GRAPH_BUCKET_NAME = "wellcomecollection-catalogue-graph"
PIPELINE_DATE = "2025-10-02"


class BaseS3Reader(BaseModel):
    bucket_name: str = GRAPH_BUCKET_NAME
    pipeline_date: str = PIPELINE_DATE

    @property
    def service_prefix(self) -> str:
        raise NotImplementedError()

    def get_uris_for_window(self, window: IncrementalWindow) -> list[str]:
        """Return the S3 URIs of all objects associated with a given incremental window"""
        raise NotImplementedError()

    def get_file_as_df(self, s3_uri: str) -> pl.DataFrame:
        """Return the contents of an S3 artefact as a Polars dataframe"""
        raise NotImplementedError()

    @property
    def windows_prefix(self) -> str:
        return f"{self.service_prefix}/{self.pipeline_date}/windows"

    def s3_uris_to_df(self, s3_uris: list[str]) -> pl.DataFrame:
        """Given a list of S3 URIs, return their contents as a single Polars dataframe."""
        dfs = []
        for s3_uri in s3_uris:
            try:
                df = self.get_file_as_df(s3_uri)
                if len(df) > 0:
                    dfs.append(df)
            except OSError as e:
                print(f"File '{s3_uri}' does not exist")

        if len(dfs) == 0:
            return pl.DataFrame()

        return pl.concat(dfs)
    
    def get_single_window_df(self, window: IncrementalWindow) -> pl.DataFrame:
        """Return all artefacts associated with a single incremental window as a Polars dataframe"""
        s3_uris = self.get_uris_for_window(window)
        df = self.s3_uris_to_df(s3_uris)
        if len(df) > 0:
            df = df.with_columns(pl.lit(window.start_time).alias("window_start"))
            df = df.with_columns(pl.lit(window.end_time).alias("window_end"))
        return df

    def dataframe_from_incremental_files(
            self, range_start: datetime | None = None, range_end: datetime | None = None
    ) -> pl.DataFrame:
        """
        Return all artefacts associated with all incremental windows as a Polars dataframe,
        supporting optional `range_start` and `range_end` parameters to filter windows.
        """
        windows = list_incremental_windows(
            self.bucket_name, f"{self.windows_prefix}/", range_start, range_end
        )
        print(
            f"Will process {len(windows):,} {self.service_prefix} time windows. Pipeline date: {self.pipeline_date}."
        )

        with ThreadPoolExecutor(max_workers=S3_PARALLELISM) as executor:
            dfs = executor.map(self.get_single_window_df, windows)

        non_empty_dfs = [df for df in dfs if len(df) > 0]
        if not non_empty_dfs:
            return pl.DataFrame()

        full_df = pl.concat(non_empty_dfs)
        print(f"Retrieved {len(full_df):,} rows")

        return full_df.sort("window_end")


class BulkLoaderReader(BaseS3Reader):
    transformer_type: TransformerType
    entity_type: EntityType

    @property
    def service_prefix(self) -> str:
        return "graph_bulk_loader"

    @property
    def file_name(self) -> str:
        return f"{self.transformer_type}__{self.entity_type}.csv"

    def get_uris_for_window(self, window: IncrementalWindow) -> list[str]:
        return [
            f"s3://{self.bucket_name}/{self.windows_prefix}/{strf_window(window)}/{self.file_name}"
        ]

    def get_file_as_df(self, s3_uri: str) -> pl.DataFrame:
        return pl.DataFrame(get_csv_from_s3(s3_uri))

    def dataframe_from_full_reindex_file(self) -> pl.DataFrame:
        full_reindex_uri = f"s3://{self.bucket_name}/{self.service_prefix}/{self.pipeline_date}/{self.file_name}"
        df = self.get_file_as_df(full_reindex_uri)
        print(f"Retrieved a full reindex bulk load file storing {len(df):,} rows")
        return df


class GraphRemoverIncrementalReader(BaseS3Reader):
    transformer_type: CatalogueTransformerType
    entity_type: EntityType

    @property
    def service_prefix(self) -> str:
        return "graph_remover_incremental"

    @property
    def file_name(self) -> str:
        return f"{self.transformer_type}__{self.entity_type}.parquet"

    def get_uris_for_window(self, window: IncrementalWindow) -> list[str]:
        return [
            f"s3://{self.bucket_name}/{self.windows_prefix}/{strf_window(window)}/deleted_ids/{self.file_name}"
        ]

    def get_file_as_df(self, s3_uri: str) -> pl.DataFrame:
        df = df_from_s3_parquet(s3_uri)
        if len(df) > 0:
            df.columns = ["id"]
        return df


class IngestorReader(BaseS3Reader):
    ingestor_type: Literal["works", "concepts"]
    index_date: str

    @property
    def service_prefix(self) -> str:
        return f"ingestor_{self.ingestor_type}"

    @property
    def windows_prefix(self) -> str:
        return f"{self.service_prefix}/{self.pipeline_date}/{self.index_date}"

    def get_uris_for_window(self, window: IncrementalWindow) -> list[str]:
        prefix = f"{self.windows_prefix}/{strf_window(window)}/"
        return list_s3_uris(self.bucket_name, prefix, ".parquet")

    def get_file_as_df(self, s3_uri: str) -> pl.DataFrame:
        return df_from_s3_parquet(s3_uri)

    def dataframe_from_full_reindex_files(self, job_id: str) -> pl.DataFrame:
        prefix = f"{self.windows_prefix}/{job_id}/"
        full_reindex_uris = list_s3_uris(self.bucket_name, prefix, ".parquet")
        df = self.s3_uris_to_df(full_reindex_uris)
        print(f"Retrieved all full reindex files storing {len(df):,} rows")
        return df        

In [ ]:
"""Example bulk loader reader"""
reader = BulkLoaderReader(
    transformer_type="catalogue_concepts",
    entity_type="nodes"
)
df = reader.dataframe_from_incremental_files()
df.filter(pl.col(":ID") == "kwyaf9w7")

In [ ]:
"""Example graph remover incremental reader"""
reader = GraphRemoverIncrementalReader(
    transformer_type="catalogue_concepts",
    entity_type="nodes"
)
range_start = datetime(2025, 10, 5)
range_end = datetime(2025, 10, 20)
df = reader.dataframe_from_incremental_files(range_start, range_end)
df.filter(pl.col("id") == "snb65nn6")

In [ ]:
"""Example ingestor reader"""
reader = IngestorReader(
    ingestor_type="concepts",
    index_date="2026-03-03"
)

df = reader.dataframe_from_incremental_files()
df.filter(pl.col("query").struct.field("id") == "taythpbh")